# Notebook 01 Dataset Preparation
**FinAlign |**

End-to-end data pipeline:  
Part A: Load HuggingFace data  Length filter  Deduplication  PII scrub  Format standardise  
Part B: Quality scoring  Stratified split  500 preference pairs  100-question eval set

## 0. Install

In [22]:
!pip install -q datasets huggingface_hub
import sys, os
from pathlib import Path
sys.path.extend(['src', '../src', os.path.abspath('src'), os.path.abspath('../src')])

PROJECT_ROOT = Path(os.path.abspath('..')) if os.path.basename(os.getcwd()) == 'notebooks' else Path(os.path.abspath('.'))
def _resolve_path(p):
    path = Path(p)
    return path if path.is_absolute() else PROJECT_ROOT / path

if 'data_pipeline' in sys.modules:
    import importlib, data_pipeline
    importlib.reload(data_pipeline)

print("Ready")

Ready



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## PART A 

### A1. Load Raw Data from HuggingFace

In [23]:
from data_pipeline import load_huggingface_data
raw_records = load_huggingface_data(cache_dir="data/raw")
print(f"Total raw records loaded: {len(raw_records)}")

# Preview
for r in raw_records[:2]:
    print("---")
    print("INSTRUCTION:", r["instruction"][:120])
    print("RESPONSE:   ", r["response"][:120])
    print("SOURCE:     ", r["source"])

[Load] Akhil-Theerthala/PersonalFinance-Reddit-QA ...
       Columns: ['category', 'subreddit', 'query', 'answer']
       Loaded: 19984 records
[Load] ceadar-ie/FinTalk-19k ...
       Columns: ['instruction', 'response', 'context', 'tag']
       Loaded: 19111 records
[Load] Merging local dataset: C:\Users\Asus\OneDrive\FinAlign\data\finalign_cleaned_dataset.jsonl
       Merged 0 additional records from local file
[Load] Total raw records: 39095
Total raw records loaded: 39095
---
INSTRUCTION: Title: "Amount Owed Too High" C. Score, but isn't!?
 Query: Hi everyone, first post!

I'm in the early stages of mortgag
RESPONSE:    ### Understanding Your Frustration  

First, let me validate your feelings—your confusion and frustration are completely
SOURCE:      reddit_qa
---
INSTRUCTION: Title: "Fixing" a credit account/report
 Query: I'm trying to help a friend of mine out who has never in his life ever o
RESPONSE:    I can hear the frustration and concern in your voice as you navigate this

### A2. Length Filter

In [24]:
from data_pipeline import length_filter
filtered, f_stats = length_filter(
    raw_records,
    min_instruction_chars=20,
    max_instruction_chars=2000,
    min_response_chars=100,
    max_response_chars=8000,
    min_response_words=20,
)
print("\nLength Filter Stats:")
for k, v in f_stats.items():
    print(f"  {k}: {v}")

[Filter] 39095 -> 37877 (removed 1218, 3.12%)

Length Filter Stats:
  before: 39095
  after: 37877
  removed: 1218
  pct_removed: 3.12
  breakdown: {'instruction_too_long': 876, 'response_too_long': 20, 'response_too_few_words': 148, 'response_too_short': 169, 'instruction_too_short': 5}


### A3. Deduplication (Exact + Near-duplicate)

In [25]:
from data_pipeline import deduplicate
deduped, d_stats = deduplicate(filtered, jaccard_threshold=0.7)
print("\nDedup Stats:")
for k, v in d_stats.items():
    print(f"  {k}: {v}")

[Dedup] Exact: removed 916 -> 36961 remain
[Dedup] Near-dup (J>=0.7): removed 707 -> 36254 remain

Dedup Stats:
  before: 37877
  exact_removed: 916
  near_dup_removed: 707
  after: 36254
  pct_removed: 4.28


### A4. PII Scrubbing

In [26]:
from data_pipeline import scrub_pii
scrubbed, pii_stats = scrub_pii(deduped)
print("\nPII Scrub Stats:", pii_stats)

# Show a before/after example if any PII was found
for orig, clean in zip(deduped[:500], scrubbed[:500]):
    if orig["instruction"] != clean["instruction"] or orig["response"] != clean["response"]:
        print("\nBEFORE:", orig["instruction"][:200])
        print("AFTER: ", clean["instruction"][:200])
        break

[PII]   Scrubbed 5181 PII instances across 36254 records
        Breakdown: {'[EMAIL]': 39, '[PHONE]': 744, '[URL]': 4334, '[USER]': 29, '[ACCOUNT]': 35}

PII Scrub Stats: {'total_replacements': 5181, 'by_type': {'[EMAIL]': 39, '[PHONE]': 744, '[URL]': 4334, '[USER]': 29, '[ACCOUNT]': 35}}

BEFORE: Title: "Fixing" a credit account/report
 Query: I'm trying to help a friend of mine out who has never in his life ever owned a credit card or had a loan in his name or any bills to pay. He got a notic
AFTER:  Title: "Fixing" a credit account/report
 Query: I'm trying to help a friend of mine out who has never in his life ever owned a credit card or had a loan in his name or any bills to pay. He got a notic


### A5. Format Standardize  {instruction, input, output}

In [27]:
from data_pipeline import standardize_format
final_a, fmt_stats = standardize_format(scrubbed)
print("\nFormat Stats:")
for k, v in fmt_stats.items():
    print(f"  {k}: {v}")
print("\nSample standardized record:")
import json
print(json.dumps(final_a[0], indent=2))

[Format] Off-topic removed: 6641 | Final: 29613

Format Stats:
  before: 36254
  off_topic_removed: 6641
  after: 29613
  pct_removed: 18.32

Sample standardized record:
{
  "instruction": "Title: \"Amount Owed Too High\" C. Score, but isn't!?\n Query: Hi everyone, first post!\n\nI'm in the early stages of mortgage applications. I got a \"we pulled your scores, so we'll show you\" sheet from a lender. \n\nMy scores are:\nEquifax: 800\nTransUnion: 782\nExperience 773\n\nI feel good about those! I'm fairly young and it continues to improve.\n\nHowever. It has an area that notes adverse factors, of which: Experian and TransUnion say \"Amount owed on accounts is too high\" and Equifax says \"Amount owed on revolving account too high.\"\n\nThis is confusing for me. Here's my stats:\n\nOne credit card, Discover, *never* over $1000, always paid twice monthly before due, credit limit $14,000. Even if a snapshot got it at a high end of $1000, that's still low, yeah?\n\nOne installment loan for 

### A6. Save Cleaned Dataset

In [28]:
import json
from pathlib import Path
from data_pipeline import _resolve_path
out_dir = _resolve_path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / "finance_sft_clean.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for r in final_a:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(final_a)} records -> {out}")
assert len(final_a) >= 5000, f"Need 5000+ pairs, got {len(final_a)}"
print("Target 5,000+ pairs: PASSED")

Saved 29613 records -> C:\Users\Asus\OneDrive\FinAlign\data\processed\finance_sft_clean.jsonl
Target 5,000+ pairs: PASSED


## PART B

### B1. Quality Scoring

In [29]:
from data_pipeline import quality_score
import random

sample = random.sample(final_a, min(1000, len(final_a)))
scores = [quality_score(r) for r in sample]
totals = [s["total"] for s in scores]

import statistics
print(f"Quality Score Distribution (0-9 scale, sample n={len(scores)}):")
print(f"  Mean  : {statistics.mean(totals):.2f}")
print(f"  Median: {statistics.median(totals):.2f}")
print(f"  Stdev : {statistics.stdev(totals):.2f}")
print(f"  Min   : {min(totals)}")
print(f"  Max   : {max(totals)}")
from collections import Counter
dist = Counter(totals)
for score in sorted(dist):
    bar = "#" * (dist[score] // 5)
    print(f"  {score}: {bar} ({dist[score]})")

Quality Score Distribution (0-9 scale, sample n=1000):
  Mean  : 6.74
  Median: 7.00
  Stdev : 2.25
  Min   : 2
  Max   : 9
  2: # (6)
  3: ################### (98)
  4: ############################### (158)
  5: ################ (82)
  6: ############ (62)
  7: ###################### (110)
  8: ##################### (109)
  9: ########################################################################### (375)


### B2. Stratified Train/Val/Test Split

In [30]:
from data_pipeline import stratified_split
train, val, test, split_stats = stratified_split(
    final_a, train_ratio=0.80, val_ratio=0.10, test_ratio=0.10,
    quality_threshold=4, seed=42,
)

print("\nSplit Stats:")
print(f"  Train : {len(train)}")
print(f"  Val   : {len(val)}")
print(f"  Test  : {len(test)}")
print("\nTopic breakdown (train):")
from collections import Counter
train_topics = Counter(r.get("_topic","") for r in train)
for t, c in sorted(train_topics.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c}")

[Split] Train=21707 | Val=2957 | Test=2957
        No leakage guaranteed. Topics: ['credit', 'income', 'debt', 'insurance', 'investing', 'budgeting', 'saving', 'tax', 'mortgage', 'retirement', 'general']

Split Stats:
  Train : 21707
  Val   : 2957
  Test  : 2957

Topic breakdown (train):
  investing: 4709
  debt: 3704
  budgeting: 2965
  tax: 2252
  mortgage: 2088
  saving: 1977
  retirement: 1148
  insurance: 1022
  credit: 1007
  income: 545
  general: 290


### B3. Generate 500 Preference Pairs

In [31]:
from data_pipeline import generate_preference_pairs
pairs = generate_preference_pairs(train, n_pairs=500, seed=42, topic_balance=True)

print(f"\nGenerated: {len(pairs)} preference pairs")
print("\n--- Sample Pairs (sanity check) ---")
import random
samples = random.sample(pairs, min(5, len(pairs)))
for i, p in enumerate(samples, 1):
    print(f"\n[Pair {i}] Topic: {p['topic']}")
    print(f"  PROMPT:   {p['prompt'][10:100]}...")
    print(f"  CHOSEN:   {p['chosen'][:150]}")
    print(f"  REJECTED: {p['rejected'][:150]}")

[Pref]  Generated 500 preference pairs across 11 topics
        budgeting: 46
        credit: 46
        debt: 46
        general: 45
        income: 45
        insurance: 45
        investing: 46
        mortgage: 45
        retirement: 45
        saving: 46
        tax: 45

Generated: 500 preference pairs

--- Sample Pairs (sanity check) ---

[Pair 1] Topic: saving
  PROMPT:   Title: Regular investing account or IRA for 5-10 year horizon investing?
 Query: Any advic...
  CHOSEN:    ### Understanding Your Situation and Goals  

First, I want to acknowledge that you're thinking carefully about how to grow your money wisely while m
  REJECTED:  ### Understanding Your Situation and Goals First, I want to acknowledge that you're thinking carefully about how to grow your money wisely while mini

[Pair 2] Topic: income
  PROMPT:   Title: Offered an opportunity to run a small project on type of my daily job.
 Query: I’ll...
  CHOSEN:    ### **Analysis of Your Situation**  

1. **Temporary Ro

### B4. Save Splits + Preference Pairs

In [32]:
from pathlib import Path
import json
from data_pipeline import _resolve_path

pref_dir = _resolve_path("data/preference_pairs")
proc_dir = _resolve_path("data/processed")
pref_dir.mkdir(parents=True, exist_ok=True)
proc_dir.mkdir(parents=True, exist_ok=True)

def _strip(rs):
    return [{k:v for k,v in r.items() if not k.startswith("_")} for r in rs]

for name, records in [("train", train), ("val", val), ("test", test)]:
    path = proc_dir / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for r in _strip(records):
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {path} ({len(records)} records)")

n_tr = int(len(pairs) * 0.8)
for name, ps in [("train_pref", pairs[:n_tr]), ("val_pref", pairs[n_tr:])]:
    path = pref_dir / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for p in ps:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"Saved {path} ({len(ps)} pairs)")

Saved C:\Users\Asus\OneDrive\FinAlign\data\processed\train.jsonl (21707 records)
Saved C:\Users\Asus\OneDrive\FinAlign\data\processed\val.jsonl (2957 records)
Saved C:\Users\Asus\OneDrive\FinAlign\data\processed\test.jsonl (2957 records)
Saved C:\Users\Asus\OneDrive\FinAlign\data\preference_pairs\train_pref.jsonl (400 pairs)
Saved C:\Users\Asus\OneDrive\FinAlign\data\preference_pairs\val_pref.jsonl (100 pairs)


### B5. Generate 100-Question Eval Set

In [33]:
from data_pipeline import generate_eval_set, _resolve_path
from pathlib import Path
import json

eval_items = generate_eval_set(test, n_questions=100, seed=42)
eval_dir = _resolve_path("data/eval_set")
eval_dir.mkdir(parents=True, exist_ok=True)
eval_file = eval_dir / "benchmark_100q.jsonl"
with open(eval_file, "w", encoding="utf-8") as f:
    for item in eval_items:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"Eval set saved: {len(eval_items)} questions")
from collections import Counter
print("Topic distribution:", dict(Counter(e["topic"] for e in eval_items)))

[Eval]  Carved out 100 eval questions
Eval set saved: 100 questions
Topic distribution: {'saving': 9, 'retirement': 9, 'budgeting': 9, 'income': 9, 'credit': 9, 'mortgage': 9, 'general': 9, 'investing': 9, 'debt': 9, 'tax': 9, 'insurance': 10}


### B6. Leakage Check

In [34]:
# Verify no prompt leakage between train and val/test
train_instr = {r["instruction"].strip().lower() for r in train}
val_instr   = {r["instruction"].strip().lower() for r in val}
test_instr  = {r["instruction"].strip().lower() for r in test}

train_val_leak  = len(train_instr & val_instr)
train_test_leak = len(train_instr & test_instr)

print(f"Train-Val overlap  : {train_val_leak}  {'PASS' if train_val_leak == 0 else 'FAIL'}")
print(f"Train-Test overlap : {train_test_leak}  {'PASS' if train_test_leak == 0 else 'FAIL'}")

Train-Val overlap  : 0  PASS
Train-Test overlap : 0  PASS


### Data Stats Report

In [35]:
import json
from data_pipeline import _resolve_path

stats = {
    "part_a": {
        "raw_count":           len(raw_records),
        "after_length_filter": len(filtered),
        "after_dedup":         len(deduped),
        "after_pii_scrub":     len(scrubbed),
        "after_format_topic":  len(final_a),
        "pct_removed":         round(100*(len(raw_records)-len(final_a))/max(1,len(raw_records)),2),
        "length_filter_breakdown": f_stats["breakdown"],
        "pii_replacements":    pii_stats,
    },
    "part_b": {
        "train": len(train), "val": len(val), "test": len(test),
        "train_pref_pairs": n_tr,
        "val_pref_pairs":   len(pairs) - n_tr,
        "eval_questions":   len(eval_items),
        "topic_breakdown":  {t:c for t,c in split_stats["topic_breakdown"].items()},
    }
}
stats_file = _resolve_path("data/data_stats_report.json")
with open(stats_file, "w") as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))

{
  "part_a": {
    "raw_count": 39095,
    "after_length_filter": 37877,
    "after_dedup": 36254,
    "after_pii_scrub": 36254,
    "after_format_topic": 29613,
    "pct_removed": 24.25,
    "length_filter_breakdown": {
      "instruction_too_long": 876,
      "response_too_long": 20,
      "response_too_few_words": 148,
      "response_too_short": 169,
      "instruction_too_short": 5
    },
    "pii_replacements": {
      "total_replacements": 5181,
      "by_type": {
        "[EMAIL]": 39,
        "[PHONE]": 744,
        "[URL]": 4334,
        "[USER]": 29,
        "[ACCOUNT]": 35
      }
    }
  },
  "part_b": {
    "train": 21707,
    "val": 2957,
    "test": 2957,
    "train_pref_pairs": 400,
    "val_pref_pairs": 100,
    "eval_questions": 100,
    "topic_breakdown": {
      "credit": {
        "total": 1368,
        "train": 1007,
        "val": 136,
        "test": 136
      },
      "income": {
        "total": 831,
        "train": 545,
        "val": 83,
        "test": 8